# 🎮 Android Game Character Modder - Google Colab

## Replace Characters, Textures & Assets in Android Games

This notebook provides a complete toolkit for modding Android games, similar to what modders like **Toasty Crunch** do for PC games. You can:

- ✅ Decompile APK files
- ✅ Extract and replace character textures/sprites
- ✅ Modify 3D models (if the game uses common formats)
- ✅ Edit game assets and resources
- ✅ Recompile and sign the modded APK

---

### ⚠️ Legal Disclaimer
This tool is for **educational purposes only**. Modding games may violate Terms of Service. Only mod games you own and never distribute modded APKs of paid games. The authors are not responsible for any misuse.

## 📋 Table of Contents

1. [Environment Setup](#setup)
2. [Upload APK](#upload)
3. [Decompile APK](#decompile)
4. [Explore Game Assets](#explore)
5. [Character/Texture Replacement](#replace)
6. [3D Model Editing](#models)
7. [Recompile & Sign APK](#recompile)
8. [Download Modded APK](#download)

---
<a name='setup'></a>
## 1. 🔧 Environment Setup

Run this cell first to install all necessary tools:

In [ ]:
#@title Install Android Modding Tools { display-mode: "form" }

import os
import sys
import subprocess
from pathlib import Path

print("🎮 Setting up Android Game Modding Environment...")
print("="*50)

# Install Java (required for apktool)
print("☕ Installing Java...")
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y default-jdk > /dev/null 2>&1

# Install apktool
print("🔧 Installing APKTool...")
!apt-get install -y apktool > /dev/null 2>&1

# Install additional tools
print("📦 Installing additional tools...")
!apt-get install -y zip unzip p7zip-full > /dev/null 2>&1
!pip install -q Pillow numpy

# Install uber-apk-signer for signing APKs
print("✍️ Installing APK Signer...")
APK_SIGNER_URL = "https://github.com/patrickfav/uber-apk-signer/releases/download/v1.3.0/uber-apk-signer-1.3.0.jar"
!wget -q {APK_SIGNER_URL} -O /content/uber-apk-signer.jar

# Install AssetStudio for Unity games (optional)
print("🎯 Setting up Unity Asset extraction tools...")
!pip install -q UnityPy

# Install additional asset tools
print("🖼️ Installing texture/model tools...")
!pip install -q texture2ddecoder kreaitch-etc1-decoder 2>/dev/null || true

# Create working directories
WORK_DIR = "/content/modding_workspace"
os.makedirs(f"{WORK_DIR}/input", exist_ok=True)
os.makedirs(f"{WORK_DIR}/output", exist_ok=True)
os.makedirs(f"{WORK_DIR}/extracted", exist_ok=True)
os.makedirs(f"{WORK_DIR}/mods", exist_ok=True)

print("\n" + "="*50)
print("✅ Environment Setup Complete!")
print("="*50)
print(f"\n📁 Working Directory: {WORK_DIR}")
print("\n🛠️ Available Tools:")
print("   • APKTool - APK decompilation/recompilation")
print("   • Uber-APK-Signer - APK signing")
print("   • UnityPy - Unity asset extraction")
print("   • Pillow - Image processing")
print("   • 7-Zip - Archive extraction")

---
<a name='upload'></a>
## 2. 📤 Upload APK

Upload the Android game APK you want to mod:

In [ ]:
#@title Upload APK File { display-mode: "form" }

from google.colab import files
import shutil
import os

WORK_DIR = "/content/modding_workspace"

print("📤 Upload your APK file...")
print("   (Click 'Choose Files' button below)\n")

uploaded = files.upload()

if uploaded:
    for filename in uploaded.keys():
        if filename.endswith('.apk'):
            apk_path = f"{WORK_DIR}/input/{filename}"
            shutil.move(filename, apk_path)
            print(f"\n✅ APK uploaded successfully!")
            print(f"📁 Location: {apk_path}")
            print(f"📊 Size: {os.path.getsize(apk_path) / (1024*1024):.2f} MB")
            
            # Store the APK path for later cells
            with open(f"{WORK_DIR}/current_apk.txt", 'w') as f:
                f.write(apk_path)
        else:
            print(f"\n⚠️ Please upload an .apk file. You uploaded: {filename}")
else:
    print("\n⚠️ No file uploaded.")

### Alternative: Use APK from URL

If you have a direct download link for the APK:

In [ ]:
#@title Download APK from URL { display-mode: "form" }
APK_URL = "" #@param {type:"string"}

if APK_URL:
    print("📥 Downloading APK...")
    !wget -q "{APK_URL}" -P /content/modding_workspace/input/
    print("✅ Download complete!")
    !ls -la /content/modding_workspace/input/
else:
    print("⚠️ Please enter an APK URL above.")

---
<a name='decompile'></a>
## 3. 🔓 Decompile APK

This will extract and decompile the APK so you can access game assets:

In [ ]:
#@title Decompile APK { display-mode: "form" }

import os
import subprocess

WORK_DIR = "/content/modding_workspace"

# Get the APK path
apk_files = [f for f in os.listdir(f"{WORK_DIR}/input") if f.endswith('.apk')]

if not apk_files:
    print("❌ No APK file found. Please upload an APK first!")
else:
    apk_path = f"{WORK_DIR}/input/{apk_files[0]}"
    apk_name = os.path.splitext(apk_files[0])[0]
    output_dir = f"{WORK_DIR}/extracted/{apk_name}"
    
    print(f"🔓 Decompiling: {apk_files[0]}")
    print("   This may take a few minutes...\n")
    
    # Use apktool to decompile
    result = subprocess.run(
        ['apktool', 'd', apk_path, '-o', output_dir, '-f'],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✅ Decompilation successful!")
        print(f"📁 Extracted to: {output_dir}")
        
        # Store for later
        with open(f"{WORK_DIR}/extracted_dir.txt", 'w') as f:
            f.write(output_dir)
            
        # Show structure
        print("\n📂 APK Structure:")
        for item in os.listdir(output_dir):
            item_path = os.path.join(output_dir, item)
            if os.path.isdir(item_path):
                print(f"   📁 {item}/")
            else:
                print(f"   📄 {item}")
    else:
        print("❌ Decompilation failed!")
        print(result.stderr)

---
<a name='explore'></a>
## 4. 🔍 Explore Game Assets

Find and explore character textures, models, and other assets:

In [ ]:
#@title Explore Asset Structure { display-mode: "form" }

import os
from pathlib import Path

WORK_DIR = "/content/modding_workspace"

# Read extracted directory
try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
except:
    print("❌ Please decompile an APK first!")
    extracted_dir = None

if extracted_dir and os.path.exists(extracted_dir):
    print("🔍 Analyzing game assets...\n")
    
    # Common asset locations
    asset_dirs = {
        'Unity Assets': ['assets', 'Assets', 'assets/bin'],
        'Textures': ['assets/textures', 'Assets/Textures', 'res/drawable', 'res/drawable-nodpi'],
        'Models': ['assets/models', 'Assets/Models', 'models'],
        'Sprites': ['assets/sprites', 'Assets/Sprites', 'sprites'],
        'Audio': ['assets/audio', 'Assets/Audio', 'res/raw'],
        'Data': ['assets/data', 'Assets/StreamingAssets', 'assets/bin/Data']
    }
    
    found_assets = {}
    
    for category, search_paths in asset_dirs.items():
        for search_path in search_paths:
            full_path = os.path.join(extracted_dir, search_path)
            if os.path.exists(full_path):
                if category not in found_assets:
                    found_assets[category] = []
                found_assets[category].append(full_path)
    
    print("="*50)
    print("📊 ASSET ANALYSIS RESULTS")
    print("="*50 + "\n")
    
    if found_assets:
        for category, paths in found_assets.items():
            print(f"\n🎯 {category}:")
            for path in paths:
                file_count = sum(1 for _, _, files in os.walk(path) for f in files)
                print(f"   📁 {path}")
                print(f"      └── {file_count} files")
    
    # Find image/texture files
    print("\n" + "="*50)
    print("🖼️ IMAGE/TEXTURE FILES FOUND")
    print("="*50 + "\n")
    
    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp', '.tga', '.webp', '.dds', '.pvr', '.etc', '.ktx']
    image_files = []
    
    for root, dirs, files in os.walk(extracted_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in image_extensions):
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, extracted_dir)
                size = os.path.getsize(full_path)
                image_files.append((rel_path, size))
    
    if image_files:
        # Sort by size (largest first - likely character textures)
        image_files.sort(key=lambda x: x[1], reverse=True)
        
        print(f"Found {len(image_files)} image files. Top 20 by size:\n")
        for i, (path, size) in enumerate(image_files[:20]):
            print(f"   {i+1}. {path}")
            print(f"      Size: {size/1024:.1f} KB")
    else:
        print("   No common image formats found.")
        print("   Assets may be in proprietary formats (Unity bundles, etc.)")
    
    # Check for Unity assets
    print("\n" + "="*50)
    print("🎯 UNITY ASSET FILES")
    print("="*50 + "\n")
    
    unity_extensions = ['.assets', '.resS', '.resource', '.bundle', '.unity3d']
    unity_files = []
    
    for root, dirs, files in os.walk(extracted_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in unity_extensions):
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, extracted_dir)
                unity_files.append(rel_path)
    
    if unity_files:
        print(f"Found {len(unity_files)} Unity asset files:\n")
        for uf in unity_files[:15]:
            print(f"   📦 {uf}")
        if len(unity_files) > 15:
            print(f"   ... and {len(unity_files) - 15} more")
    else:
        print("   No Unity asset files found.")
else:
    print("❌ No extracted APK found. Please decompile first!")

---
<a name='replace'></a>
## 5. 🎭 Character/Texture Replacement

### Option A: Replace Simple Textures (PNG/JPG)

For games that use standard image formats:

In [ ]:
#@title Browse Texture Files { display-mode: "form" }

import os
from IPython.display import Image, display
import ipywidgets as widgets

WORK_DIR = "/content/modding_workspace"

# Find all image files
def find_images(base_dir):
    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp', '.webp']
    images = []
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in image_extensions):
                images.append(os.path.join(root, file))
    return images

try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
    
    images = find_images(extracted_dir)
    
    if images:
        # Show first few images
        print(f"🖼️ Found {len(images)} image files. Showing preview of first 10:\n")
        
        for i, img_path in enumerate(images[:10]):
            rel_path = os.path.relpath(img_path, extracted_dir)
            print(f"\n[{i+1}] {rel_path}")
            print(f"    Full path: {img_path}")
            try:
                display(Image(filename=img_path, width=200))
            except:
                print("    (Cannot display this image format)")
    else:
        print("No standard image files found. Assets may be in proprietary formats.")
        print("Try the Unity Asset Extraction section below.")
        
except Exception as e:
    print(f"Error: {e}")
    print("Please decompile an APK first!")

In [ ]:
#@title Upload Replacement Texture { display-mode: "form" }

from google.colab import files
import os

WORK_DIR = "/content/modding_workspace"

# First, specify which texture to replace
TARGET_TEXTURE = "" #@param {type:"string"}

print("📤 Upload your replacement texture (PNG/JPG)...\n")
print("💡 TIP: Make sure your replacement has the same dimensions as the original!")

uploaded = files.upload()

if uploaded:
    replacement_file = list(uploaded.keys())[0]
    replacement_path = f"{WORK_DIR}/mods/{replacement_file}"
    
    import shutil
    shutil.move(replacement_file, replacement_path)
    
    print(f"\n✅ Replacement texture uploaded!")
    print(f"📁 Location: {replacement_path}")
    
    # Get image dimensions
    from PIL import Image
    img = Image.open(replacement_path)
    print(f"📐 Dimensions: {img.size[0]}x{img.size[1]}")
    
    # Store for later
    with open(f"{WORK_DIR}/replacement_texture.txt", 'w') as f:
        f.write(replacement_path)

In [ ]:
#@title Apply Texture Replacement { display-mode: "form" }

import os
import shutil
from PIL import Image

WORK_DIR = "/content/modding_workspace"

# Parameters
ORIGINAL_TEXTURE_PATH = "" #@param {type:"string"}
REPLACEMENT_TEXTURE_PATH = "" #@param {type:"string"}

# Or use uploaded files
try:
    with open(f"{WORK_DIR}/replacement_texture.txt", 'r') as f:
        if not REPLACEMENT_TEXTURE_PATH:
            REPLACEMENT_TEXTURE_PATH = f.read().strip()
except:
    pass

try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
except:
    extracted_dir = None

if not ORIGINAL_TEXTURE_PATH or not REPLACEMENT_TEXTURE_PATH:
    print("❌ Please specify both original and replacement texture paths!")
elif not extracted_dir:
    print("❌ No decompiled APK found!")
else:
    # Construct full path
    if not ORIGINAL_TEXTURE_PATH.startswith('/'):
        full_original = os.path.join(extracted_dir, ORIGINAL_TEXTURE_PATH)
    else:
        full_original = ORIGINAL_TEXTURE_PATH
    
    if os.path.exists(full_original) and os.path.exists(REPLACEMENT_TEXTURE_PATH):
        # Backup original
        backup_path = full_original + ".backup"
        if not os.path.exists(backup_path):
            shutil.copy2(full_original, backup_path)
            print(f"💾 Backed up original to: {backup_path}")
        
        # Open and process replacement
        replacement = Image.open(REPLACEMENT_TEXTURE_PATH)
        original = Image.open(full_original)
        
        # Resize if needed (with warning)
        if replacement.size != original.size:
            print(f"⚠️ Size mismatch! Original: {original.size}, Replacement: {replacement.size}")
            print("   Resizing replacement to match original...")
            replacement = replacement.resize(original.size, Image.LANCZOS)
        
        # Convert mode if necessary
        if replacement.mode != original.mode:
            print(f"   Converting color mode: {replacement.mode} -> {original.mode}")
            replacement = replacement.convert(original.mode)
        
        # Save replacement
        replacement.save(full_original)
        print(f"\n✅ Texture replaced successfully!")
        print(f"📁 File: {full_original}")
    else:
        print(f"❌ File not found!")
        if not os.path.exists(full_original):
            print(f"   Original: {full_original}")
        if not os.path.exists(REPLACEMENT_TEXTURE_PATH):
            print(f"   Replacement: {REPLACEMENT_TEXTURE_PATH}")

### Option B: Extract Unity Assets

For Unity-based games with bundled assets:

In [ ]:
#@title Extract Unity Assets { display-mode: "form" }

import os
import UnityPy
from PIL import Image
import io

WORK_DIR = "/content/modding_workspace"

# Find Unity asset files
try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
    
    unity_extensions = ['.assets', '.resS', '.resource', '.bundle', '.unity3d']
    unity_files = []
    
    for root, dirs, files in os.walk(extracted_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in unity_extensions):
                unity_files.append(os.path.join(root, file))
    
    if not unity_files:
        print("❌ No Unity asset files found in this APK.")
    else:
        print(f"🎯 Found {len(unity_files)} Unity asset files")
        print("📦 Extracting textures and sprites...\n")
        
        # Create output directory
        unity_extract_dir = f"{WORK_DIR}/unity_extracted"
        os.makedirs(f"{unity_extract_dir}/textures", exist_ok=True)
        os.makedirs(f"{unity_extract_dir}/sprites", exist_ok=True)
        
        texture_count = 0
        sprite_count = 0
        
        for asset_file in unity_files:
            try:
                env = UnityPy.load(asset_file)
                
                for obj in env.objects:
                    try:
                        # Extract textures
                        if obj.type.name == "Texture2D":
                            data = obj.read()
                            if data.m_Width > 64:  # Skip tiny textures
                                img = data.image
                                if img:
                                    name = data.m_Name or f"texture_{texture_count}"
                                    # Sanitize name
                                    name = "".join(c if c.isalnum() or c in "._-" else "_" for c in name)
                                    output_path = f"{unity_extract_dir}/textures/{name}.png"
                                    img.save(output_path)
                                    texture_count += 1
                                    if texture_count <= 20:
                                        print(f"   🖼️ {name}.png ({data.m_Width}x{data.m_Height})")
                        
                        # Extract sprites
                        elif obj.type.name == "Sprite":
                            data = obj.read()
                            img = data.image
                            if img:
                                name = data.m_Name or f"sprite_{sprite_count}"
                                name = "".join(c if c.isalnum() or c in "._-" else "_" for c in name)
                                output_path = f"{unity_extract_dir}/sprites/{name}.png"
                                img.save(output_path)
                                sprite_count += 1
                    
                    except Exception as e:
                        pass  # Skip problematic objects
                            
            except Exception as e:
                pass  # Skip problematic files
        
        print(f"\n" + "="*50)
        print(f"✅ EXTRACTION COMPLETE!")
        print(f"   🖼️ Textures: {texture_count}")
        print(f"   🎭 Sprites: {sprite_count}")
        print(f"   📁 Location: {unity_extract_dir}")
        print("="*50)
        
        # Store path for later
        with open(f"{WORK_DIR}/unity_extract_dir.txt", 'w') as f:
            f.write(unity_extract_dir)
            
except Exception as e:
    print(f"Error: {e}")
    print("Please decompile an APK first!")

In [ ]:
#@title Preview Extracted Unity Textures { display-mode: "form" }

import os
from IPython.display import Image, display

WORK_DIR = "/content/modding_workspace"

try:
    with open(f"{WORK_DIR}/unity_extract_dir.txt", 'r') as f:
        unity_dir = f.read().strip()
    
    # Get all textures
    texture_dir = f"{unity_dir}/textures"
    sprites_dir = f"{unity_dir}/sprites"
    
    all_images = []
    for d in [texture_dir, sprites_dir]:
        if os.path.exists(d):
            for f in os.listdir(d):
                if f.endswith('.png'):
                    all_images.append(os.path.join(d, f))
    
    # Sort by size (largest first)
    all_images.sort(key=lambda x: os.path.getsize(x), reverse=True)
    
    print(f"🖼️ Showing top 15 textures by size:\n")
    
    for i, img_path in enumerate(all_images[:15]):
        name = os.path.basename(img_path)
        size = os.path.getsize(img_path) / 1024
        print(f"\n[{i+1}] {name} ({size:.1f} KB)")
        display(Image(filename=img_path, width=250))
        
except Exception as e:
    print(f"Error: {e}")
    print("Extract Unity assets first!")

---
<a name='models'></a>
## 6. 🗿 3D Model Editing

For games with 3D character models:

In [ ]:
#@title Find 3D Model Files { display-mode: "form" }

import os

WORK_DIR = "/content/modding_workspace"

try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
    
    # Common 3D model formats
    model_extensions = ['.fbx', '.obj', '.gltf', '.glb', '.dae', '.blend', '.mb', '.ma', '.3ds', '.x']
    
    # Unity-specific model containers
    unity_model_extensions = ['.assets', '.bundle', '.unity3d']
    
    print("🔍 Searching for 3D model files...\n")
    
    found_models = []
    
    for root, dirs, files in os.walk(extracted_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in model_extensions):
                found_models.append(os.path.join(root, file))
    
    if found_models:
        print(f"🗿 Found {len(found_models)} 3D model files:\n")
        for model in found_models:
            rel_path = os.path.relpath(model, extracted_dir)
            size = os.path.getsize(model) / 1024
            print(f"   📦 {rel_path} ({size:.1f} KB)")
    else:
        print("❌ No common 3D model formats found.")
        print("\n💡 Models are likely inside Unity asset bundles.")
        print("   Try extracting them with UnityPy or AssetStudio.")
        print("\n   For character model replacement, you typically need to:")
        print("   1. Extract the model from Unity assets")
        print("   2. Edit in Blender/Maya")
        print("   3. Re-import using custom tools")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
#@title Extract 3D Models from Unity Assets { display-mode: "form" }

import os
import UnityPy

WORK_DIR = "/content/modding_workspace"

try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
    
    # Find Unity asset files
    unity_extensions = ['.assets', '.bundle', '.unity3d']
    unity_files = []
    
    for root, dirs, files in os.walk(extracted_dir):
        for file in files:
            if any(file.lower().endswith(ext) for ext in unity_extensions):
                unity_files.append(os.path.join(root, file))
    
    if not unity_files:
        print("❌ No Unity asset files found.")
    else:
        print(f"🗿 Extracting 3D models from Unity assets...\n")
        
        model_dir = f"{WORK_DIR}/extracted_models"
        os.makedirs(model_dir, exist_ok=True)
        
        mesh_count = 0
        
        for asset_file in unity_files:
            try:
                env = UnityPy.load(asset_file)
                
                for obj in env.objects:
                    try:
                        # Extract meshes
                        if obj.type.name == "Mesh":
                            data = obj.read()
                            name = data.m_Name or f"mesh_{mesh_count}"
                            
                            # Export as OBJ
                            try:
                                mesh_data = data.export()
                                if mesh_data:
                                    output_path = f"{model_dir}/{name}.obj"
                                    with open(output_path, 'wb') as f:
                                        f.write(mesh_data)
                                    mesh_count += 1
                                    print(f"   🗿 {name}.obj")
                            except:
                                pass
                    
                    except Exception as e:
                        pass
                            
            except Exception as e:
                pass
        
        print(f"\n{'='*50}")
        print(f"✅ Extracted {mesh_count} 3D meshes")
        print(f"📁 Location: {model_dir}")
        print("="*50)
        
        if mesh_count == 0:
            print("\n💡 TIP: Many games use proprietary mesh formats.")
            print("   For complex model replacement, consider using:")
            print("   • AssetStudio (Windows) - https://github.com/Perfare/AssetStudio")
            print("   • UABE (Unity Assets Bundle Extractor)")
        
except Exception as e:
    print(f"Error: {e}")

---
<a name='recompile'></a>
## 7. 📦 Recompile & Sign APK

After making your modifications, recompile and sign the APK:

In [ ]:
#@title Recompile Modified APK { display-mode: "form" }

import os
import subprocess

WORK_DIR = "/content/modding_workspace"

try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
    
    apk_name = os.path.basename(extracted_dir)
    output_apk = f"{WORK_DIR}/output/{apk_name}_modded.apk"
    
    print(f"📦 Recompiling APK...")
    print(f"   Source: {extracted_dir}")
    print(f"   Output: {output_apk}\n")
    
    # Use apktool to recompile
    result = subprocess.run(
        ['apktool', 'b', extracted_dir, '-o', output_apk],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✅ Recompilation successful!")
        print(f"📁 APK created: {output_apk}")
        print(f"📊 Size: {os.path.getsize(output_apk) / (1024*1024):.2f} MB")
        
        # Store for signing
        with open(f"{WORK_DIR}/unsigned_apk.txt", 'w') as f:
            f.write(output_apk)
    else:
        print("❌ Recompilation failed!")
        print(result.stderr)
        print("\n💡 Common issues:")
        print("   • Corrupted resources - try without modification first")
        print("   • Large files - may need to adjust apktool settings")
        
except Exception as e:
    print(f"Error: {e}")
    print("Please decompile and modify an APK first!")

In [ ]:
#@title Sign APK { display-mode: "form" }

import os
import subprocess

WORK_DIR = "/content/modding_workspace"

try:
    with open(f"{WORK_DIR}/unsigned_apk.txt", 'r') as f:
        unsigned_apk = f.read().strip()
    
    if not os.path.exists(unsigned_apk):
        print("❌ Unsigned APK not found!")
    else:
        print("✍️ Signing APK with uber-apk-signer...\n")
        
        # Use uber-apk-signer
        result = subprocess.run(
            ['java', '-jar', '/content/uber-apk-signer.jar', '--apks', unsigned_apk, '--out', f"{WORK_DIR}/output"],
            capture_output=True,
            text=True
        )
        
        print(result.stdout)
        
        if result.returncode == 0:
            # Find the signed APK
            for f in os.listdir(f"{WORK_DIR}/output"):
                if f.endswith('-aligned-debugSigned.apk'):
                    signed_apk = f"{WORK_DIR}/output/{f}"
                    print(f"\n{'='*50}")
                    print("✅ APK SIGNED SUCCESSFULLY!")
                    print(f"{'='*50}")
                    print(f"\n📁 Signed APK: {signed_apk}")
                    print(f"📊 Size: {os.path.getsize(signed_apk) / (1024*1024):.2f} MB")
                    print("\n🎮 You can now install this APK on your Android device!")
                    print("   (Make sure 'Unknown Sources' is enabled)")
                    
                    # Store for download
                    with open(f"{WORK_DIR}/signed_apk.txt", 'w') as f:
                        f.write(signed_apk)
                    break
        else:
            print("❌ Signing failed!")
            print(result.stderr)
            
except Exception as e:
    print(f"Error: {e}")
    print("Please recompile an APK first!")

---
<a name='download'></a>
## 8. ⬇️ Download Modded APK

In [ ]:
#@title Download Modded APK { display-mode: "form" }

import os
from google.colab import files

WORK_DIR = "/content/modding_workspace"

# Find signed APK
signed_apk = None
for f in os.listdir(f"{WORK_DIR}/output"):
    if 'Signed' in f or 'signed' in f:
        signed_apk = f"{WORK_DIR}/output/{f}"
        break

if not signed_apk:
    # Try any APK in output
    for f in os.listdir(f"{WORK_DIR}/output"):
        if f.endswith('.apk'):
            signed_apk = f"{WORK_DIR}/output/{f}"
            break

if signed_apk and os.path.exists(signed_apk):
    print(f"⬇️ Downloading: {os.path.basename(signed_apk)}")
    files.download(signed_apk)
else:
    print("❌ No signed APK found!")
    print("\nAvailable files in output directory:")
    output_dir = f"{WORK_DIR}/output"
    if os.path.exists(output_dir):
        for f in os.listdir(output_dir):
            print(f"   📄 {f}")

In [ ]:
#@title Download All Extracted Assets { display-mode: "form" }

import os
import shutil
from google.colab import files

WORK_DIR = "/content/modding_workspace"

# Create a zip of all extracted assets
zip_path = f"{WORK_DIR}/extracted_assets.zip"

print("📦 Creating archive of extracted assets...\n")

# Zip Unity extracted assets
unity_dir = f"{WORK_DIR}/unity_extracted"
if os.path.exists(unity_dir):
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', unity_dir)
    print(f"✅ Created: {zip_path}")
    print(f"📊 Size: {os.path.getsize(zip_path) / (1024*1024):.2f} MB")
    files.download(zip_path)
else:
    print("❌ No extracted Unity assets found.")
    print("   Run the 'Extract Unity Assets' cell first.")

---
## 🎯 Advanced: Batch Texture Replacement

Replace multiple textures at once:

In [ ]:
#@title Batch Replace Textures { display-mode: "form" }

import os
import shutil
from PIL import Image
from google.colab import files

WORK_DIR = "/content/modding_workspace"

# Upload multiple replacement textures
print("📤 Upload replacement textures (PNG/JPG)")
print("   Filenames should match the original texture names\n")

uploaded = files.upload()

if uploaded:
    try:
        with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
            extracted_dir = f.read().strip()
        
        print(f"\n🔄 Processing {len(uploaded)} replacement textures...\n")
        
        replaced = 0
        not_found = []
        
        for filename, content in uploaded.items():
            # Save uploaded file
            upload_path = f"{WORK_DIR}/mods/{filename}"
            with open(upload_path, 'wb') as f:
                f.write(content)
            
            # Search for matching texture in extracted APK
            base_name = os.path.splitext(filename)[0]
            found = False
            
            for root, dirs, files in os.walk(extracted_dir):
                for f in files:
                    if base_name.lower() in f.lower():
                        original_path = os.path.join(root, f)
                        
                        try:
                            # Load and resize
                            replacement = Image.open(upload_path)
                            original = Image.open(original_path)
                            
                            if replacement.size != original.size:
                                replacement = replacement.resize(original.size, Image.LANCZOS)
                            
                            if replacement.mode != original.mode:
                                replacement = replacement.convert(original.mode)
                            
                            # Backup and replace
                            backup_path = original_path + ".backup"
                            if not os.path.exists(backup_path):
                                shutil.copy2(original_path, backup_path)
                            
                            replacement.save(original_path)
                            
                            rel_path = os.path.relpath(original_path, extracted_dir)
                            print(f"   ✅ Replaced: {f}")
                            print(f"      Location: {rel_path}")
                            replaced += 1
                            found = True
                            break
                        except Exception as e:
                            print(f"   ❌ Error processing {f}: {e}")
                
                if found:
                    break
            
            if not found:
                not_found.append(filename)
        
        print(f"\n{'='*50}")
        print(f"📊 BATCH REPLACEMENT COMPLETE")
        print(f"{'='*50}")
        print(f"   ✅ Replaced: {replaced}")
        print(f"   ❌ Not found: {len(not_found)}")
        
        if not_found:
            print(f"\n   Textures not found:")
            for nf in not_found:
                print(f"      • {nf}")
        
    except Exception as e:
        print(f"Error: {e}")

---
## 🔧 Utility Functions

In [ ]:
#@title Restore Original Textures (Undo Changes) { display-mode: "form" }

import os
import shutil

WORK_DIR = "/content/modding_workspace"

try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
    
    print("🔄 Restoring original textures from backups...\n")
    
    restored = 0
    
    for root, dirs, files in os.walk(extracted_dir):
        for f in files:
            if f.endswith('.backup'):
                backup_path = os.path.join(root, f)
                original_path = backup_path[:-7]  # Remove .backup
                
                shutil.copy2(backup_path, original_path)
                os.remove(backup_path)
                
                rel_path = os.path.relpath(original_path, extracted_dir)
                print(f"   ✅ Restored: {rel_path}")
                restored += 1
    
    print(f"\n{'='*50}")
    print(f"✅ Restored {restored} original files")
    print("="*50)
    
except Exception as e:
    print(f"Error: {e}")

In [ ]:
#@title List All Modified Files { display-mode: "form" }

import os

WORK_DIR = "/content/modding_workspace"

try:
    with open(f"{WORK_DIR}/extracted_dir.txt", 'r') as f:
        extracted_dir = f.read().strip()
    
    print("📋 Modified files (have backups):\n")
    
    found = False
    for root, dirs, files in os.walk(extracted_dir):
        for f in files:
            if f.endswith('.backup'):
                original = f[:-7]
                rel_path = os.path.relpath(os.path.join(root, original), extracted_dir)
                print(f"   📝 {rel_path}")
                found = True
    
    if not found:
        print("   No modifications found.")
        
except Exception as e:
    print(f"Error: {e}")

In [ ]:
#@title Clean Workspace { display-mode: "form" }

import os
import shutil

WORK_DIR = "/content/modding_workspace"

print("🧹 Cleaning workspace...\n")

# Keep input and output, clean extracted
dirs_to_clean = ['extracted', 'unity_extracted', 'extracted_models']

for d in dirs_to_clean:
    path = f"{WORK_DIR}/{d}"
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"   🗑️ Removed: {d}/")

# Clean temp files
for f in ['current_apk.txt', 'extracted_dir.txt', 'unsigned_apk.txt', 'signed_apk.txt', 
n          'replacement_texture.txt', 'unity_extract_dir.txt']:
    path = f"{WORK_DIR}/{f}"
    if os.path.exists(path):
        os.remove(path)

print("\n✅ Workspace cleaned!")
print("   Preserved: input/, output/, mods/")

---
## 📚 Tips & Resources

### Best Practices for Character Modding:

1. **Texture Replacement Tips:**
   - Match original dimensions exactly for best results
   - Use PNG format for transparency support
   - Keep a backup of the original APK

2. **For Unity Games:**
   - Textures are usually in `.assets` files
   - Use AssetStudio on Windows for advanced extraction
   - Some games encrypt assets - may require additional tools

3. **Common Game Engines:**
   - **Unity**: Most common, use UnityPy or AssetStudio
   - **Unreal Engine**: Assets in `.pak` files, use QuickBMS
   - **Cocos2d**: PNG sprites, easy to replace
   - **Custom Engines**: May need game-specific tools

### Useful External Tools:

- **AssetStudio** - Advanced Unity asset extraction
- **UABE** - Unity Assets Bundle Extractor
- **QuickBMS** - Multi-format archive extractor
- **TexturePacker** - Sprite sheet unpacker
- **Blender** - 3D model editing

---

## 🎬 Like Toasty Crunch?

This notebook provides similar modding capabilities for Android games that PC modders use. For more advanced features:

- Model swapping (replacing entire 3D models)
- Animation editing
- Script modification

You may need game-specific tools and more advanced workflows. Join modding communities like:

- r/APKModding on Reddit
- XDA Developers Forums
- Nexus Mods (for some Android games)

---

**Happy Modding! 🎮**